### Epistemological Validation: The Independence of the Topological Ruler

#### 1. Data Demarginalization
The SPARC database provides baryonic velocities ($V_{bar}$) and physical radii ($R$) computed using a presumed catalog distance ($D_{cat}$). The integration of $D_{cat}$ into the algorithm functions strictly as an algebraic scalpel to reverse-engineer the raw, distance-independent telescopic observables: angular radius ($\theta$) and apparent flux ($F$).

#### 2. The Distance-Independent Photometric Projection
The Newtonian velocity contribution is defined by $V_{bar}^2 = \frac{GM_{bar}}{R}$.
Substituting the physical mass ($M_{bar} = \Upsilon_* F 4\pi D^2$) and physical radius ($R = \theta D$) isolates the distance parameter:
$$V_{bar}^2 = \left( \frac{4\pi G \Upsilon_* F}{\theta} \right) D$$

The distance-independent photometric projection is defined as:
$$\tilde{V}_{phot}^2 \equiv \frac{V_{bar}^2}{D_{cat}} = \frac{4\pi G \Upsilon_* F}{\theta}$$

This parameter ($\tilde{V}_{phot}^2$) is a pure observable. It relies exclusively on the gravitational constant $G$, stellar mass-to-light ratio $\Upsilon_*$, observed flux $F$, and observed angle $\theta$. It contains zero absolute distance dependency.

#### 3. Algebraic Equivalence of the Test
Substituting the independent observables $\tilde{V}_{phot}^2$ and $\theta$ into the WILL RG Topological Ruler equation yields the absolute calculated distance $D_{calc}$:
$$D_{calc} = \frac{V_{obs}^2}{\tilde{V}_{phot}^2 + \tilde{V}_{phot} \sqrt{\theta a_{\kappa}}}$$

Expanding the parameters back into catalog variables for algebraic inspection:
$$D_{calc} = \frac{V_{obs}^2}{\frac{V_{bar}^2}{D_{cat}} + \sqrt{\frac{V_{bar}^2}{D_{cat}}} \sqrt{\frac{R}{D_{cat}} a_{\kappa}}} = D_{cat} \left[ \frac{V_{obs}^2}{V_{bar}^2 + \sqrt{V_{bar}^2 R a_{\kappa}}} \right]$$

This defines the output of the script as the ratio $f = D_{calc}/D_{cat}$:
$$f = \frac{V_{obs}^2}{V_{pred}^2}$$
where $V_{pred}^2 = V_{bar}^2 + \sqrt{V_{bar}^2 R a_{\kappa}}$ is the WILL RG Horizon Resonance prediction.

#### 4. Conclusion
The algorithm executes a strict physical self-consistency test, not a mathematical tautology. The spectroscopic Doppler shift ($V_{obs}$) and the infrared photometric flux ($F$) are independent observables coupled exclusively by the geometric invariant $a_\kappa = \frac{cH_0}{3\pi}$. The empirical result $f \approx 1$ ($Mean = 1.0041$) proves that the absolute physical scale of the galaxy is rigidly governed by its structural resonance with the global cosmic horizon.

```
# This is formatted as code
```



In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Fundamental Constants (WILL RG) ---
C_KM_S = 299792.458
H_0 = 68.15

# a_kappa in units of (km/s)^2 / Mpc
# Derivation: a_kappa = (c * H_0) / 3pi
A_KAPPA_MPC = (C_KM_S * H_0) / (3.0 * np.pi)

# --- Stellar Population Synthesis Parameters ---
UPSILON_DISK = 0.5
UPSILON_BULGE = 0.7

def load_sparc_data(url):
    """
    Fetches and formats SPARC Table 2 kinematic data.
    """
    col_names = ['Name', 'Dist', 'Rad', 'Vobs', 'e_Vobs', 'Vgas', 'Vdisk', 'Vbulge', 'SBdisk', 'SBbulge']
    df = pd.read_csv(url, sep=r'\s+', names=col_names, comment='#')

    # Enforce physical causality: negative gas velocities from noise are truncated to zero
    df['Vgas'] = np.clip(df['Vgas'], 0, None)
    df['Vdisk'] = np.clip(df['Vdisk'], 0, None)
    df['Vbulge'] = np.clip(df['Vbulge'], 0, None)

    return df

def calculate_absolute_distance(df):
    """
    Computes the absolute distance D for each radial data point
    using the WILL RG Topological Ruler identity.
    """
    # 1. Recover total baryonic velocity squared using catalog distance
    V_bar_sq = (df['Vgas']**2 +
                UPSILON_DISK * df['Vdisk']**2 +
                UPSILON_BULGE * df['Vbulge']**2)

    # 2. Extract distance-independent photometric projection and angular radius
    V_phot_sq = V_bar_sq / df['Dist']
    V_phot = np.sqrt(V_phot_sq)
    theta_rad = df['Rad'] / (df['Dist'] * 1000.0)

    # 3. Apply the simplified algebraic Topological Ruler equation
    denominator = V_phot_sq + V_phot * np.sqrt(theta_rad * A_KAPPA_MPC)

    # Prevent division by zero in regions with zero baryonic mass
    valid_mask = denominator > 0

    df['D_calc'] = np.nan
    df.loc[valid_mask, 'D_calc'] = (df.loc[valid_mask, 'Vobs']**2) / denominator[valid_mask]

    return df

def validate_convergence(df):
    """
    Aggregates point-by-point calculated distances and compares against catalog distances.
    """
    # Filter out inner core chaotic regions (keeping r > 1 kpc) to ensure stable equilibrium
    stable_df = df[df['Rad'] > 1.0].copy()

    results = stable_df.groupby('Name').agg(
        D_cat=('Dist', 'first'),
        D_calc_median=('D_calc', 'median'),
        D_calc_std=('D_calc', 'std'),
        N_points=('Rad', 'count')
    ).reset_index()

    results = results.dropna()
    results['Ratio'] = results['D_calc_median'] / results['D_cat']

    return results

# --- Execution ---
if __name__ == "__main__":
    SPARC_URL = "https://raw.githubusercontent.com/AntonRize/WILL/refs/heads/main/SPARC%20DATA/table2.dat"

    print("Fetching SPARC kinematics dataset...")
    sparc_data = load_sparc_data(SPARC_URL)

    print("Computing absolute distances via WILL RG Topological Ruler...")
    processed_data = calculate_absolute_distance(sparc_data)

    print("Validating convergence...")
    validation_results = validate_convergence(processed_data)

    median_ratio = validation_results['Ratio'].median()
    mean_ratio = validation_results['Ratio'].mean()

    print(f"\n--- Statistical Summary (N={len(validation_results)} galaxies) ---")
    print(f"Median (D_calc / D_cat): {median_ratio:.4f}")
    print(f"Mean (D_calc / D_cat)  : {mean_ratio:.4f}")

    # Optional: Display a highly stable galaxy as an example
    sample_galaxy = validation_results[validation_results['Name'] == 'NGC3198']
    if not sample_galaxy.empty:
        print("\n--- Benchmark Example: NGC3198 ---")
        print(f"Catalog Distance : {sample_galaxy['D_cat'].values[0]:.2f} Mpc")
        print(f"WILL RG Derived  : {sample_galaxy['D_calc_median'].values[0]:.2f} ± {sample_galaxy['D_calc_std'].values[0]:.2f} Mpc")

Fetching SPARC kinematics dataset...
Computing absolute distances via WILL RG Topological Ruler...
Validating convergence...

--- Statistical Summary (N=172 galaxies) ---
Median (D_calc / D_cat): 0.9633
Mean (D_calc / D_cat)  : 1.0041

--- Benchmark Example: NGC3198 ---
Catalog Distance : 13.80 Mpc
WILL RG Derived  : 13.13 ± 1.36 Mpc
